# Dark energy model comparison

This notebook compares background dark-energy densities for multiple models
using CLASS and MochiCLASS inputs. Adjust the model lists to explore other
w0/wa combinations or expansion prescriptions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from classy import Class
import scienceplots

plt.style.use(['science', 'bright', 'vpedre'])


In [ ]:
deltalna = 7.5e-4
lna_start = -5
lna_end = 0.25
N = int((lna_end - lna_start) / deltalna) + 1

lna_smg =   np.linspace(lna_start, lna_end, N)
DeltaM2 =   np.linspace(1e-4, 1e-4, N)
Dkin =      np.linspace(1e-4, 1e-4, N)
cs2 =       np.linspace(1., 1., N)

# Stack the arrays column-wise and save
# This file is used by the stable_params gravity model.
data = np.column_stack((lna_smg, DeltaM2, Dkin, cs2))
np.savetxt("../stable_params_input/gr.dat", data, delimiter=" ")


In [ ]:
base = {
    # Reference Cosmology from EuclidEmuII
    'H0'      : 67,
    'Omega_b' : 0.049,
    'Omega_cdm' : 0.27,

    'A_s' : 2.1e-9,
    'n_s' : 0.96,
    'alpha_s' : 0,
    'k_pivot' : 0.05,

    'output':'tCl mPk dTk vTk',
    'z_max_pk': 3,
}


In [ ]:
def make_class_w0wa(w0, wa):
    params = dict(base)
    params.update({
        # Disable cosmological constant
        'Omega_Lambda': 0,
        # Dark energy fluid parameters
        'w0_fld' : w0,
        'wa_fld' : wa,
        'cs2_fld': 1.,
    })
    return params


def make_mochi_w0wa(w0, wa, root):
    params = dict(base)
    params.update({
        # Disable cosmological constant and DE fluid
        'Omega_Lambda': 0,
        'Omega_fld': 0,
        # Enable mochiclass
        'Omega_smg': -1,
        # Choose the model
        'root': root,
        'gravity_model': 'stable_params',
        'smg_file_name': '../stable_params_input/gr.dat',
        'parameters_smg': '1e-10',
        'expansion_model': 'w0wa',
        'expansion_smg': f'0.67, {w0}, {wa}',  # Lambda, w0, wa
        # CLASS evolver
        'method_qs_smg': 'automatic',
        'method_gr_smg': 'on',
        'z_gr_smg': 99.,
    })
    return params


In [ ]:
lcdm = dict(base)

models = [
    ('lcdm', lcdm),
    ('w0wa_class_w0_m1_wa_0', make_class_w0wa(-1.0, 0.0)),
    ('w0wa_class_w0_m09_wa_p02', make_class_w0wa(-0.9, 0.2)),
    ('w0wa_class_w0_m11_wa_m02', make_class_w0wa(-1.1, -0.2)),
    ('w0wa_mochi_w0_m1_wa_0', make_mochi_w0wa(-1.0, 0.0, 'output/w0wa_mochi_w0_m1_wa_0')),
    ('w0wa_mochi_w0_m09_wa_p02', make_mochi_w0wa(-0.9, 0.2, 'output/w0wa_mochi_w0_m09_wa_p02')),
    ('w0wa_mochi_w0_m11_wa_m02', make_mochi_w0wa(-1.1, -0.2, 'output/w0wa_mochi_w0_m11_wa_m02')),
]


In [ ]:
cosmo = {}
rslts = {}

for name, params in models:
    print(f'Computing model: {name}', end='', flush=True)
    cosmo[name] = Class()
    cosmo[name].set(params)
    cosmo[name].compute()
    rslts[name] = cosmo[name].get_background()
print()


In [ ]:
def pick_rho_key(bk):
    for key in ['(.)rho_lambda', '(.)rho_fld', '(.)rho_smg']:
        if key in bk:
            return key
    raise KeyError('No dark-energy density key found in background output')


fig, ax = plt.subplots()

for name, _ in models:
    bk = rslts[name]
    a = 1.0 / (1.0 + bk['z'])
    rho_key = pick_rho_key(bk)
    ax.semilogx(a, bk[rho_key], label=f'{name} ({rho_key})')

ax.set_ylabel('DE density')
ax.set_xlabel('Scale Factor $a$')
ax.legend()
